<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/An_Encoder_Decoder_Network_for_Neural_Machine_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Hugging Face Pipelines

In [ ]:
from transformers import pipeline

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline(
    task="sentiment-analysis",
    model=model_name,
    truncation=True,
    max_length=512
)

texts = [
    "This was a great movie!",
    "This was not a great movie!",
    "The acting was amazing but the story was boring."
]

results = classifier(texts)

print(results)

In [ ]:
classifier("I am from the USA")

In [ ]:

classifier("I am from Iraq")

In [ ]:
ner = pipeline(
    "ner",
    aggregation_strategy="simple"
)

text = "She works as an AI Engineer  and studies NLP with Hugging Face."

result = ner(text)

print(result)

In [ ]:
generator = pipeline(
    "text-generation",
    model="gpt2"
)

prompt = "In the future, artificial intelligence will"

result = generator(
    prompt,
    max_new_tokens=50,
    num_return_sequences=1
)

print(result[0]["generated_text"])

In [ ]:
fill_mask = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

text = "The capital of France is [MASK]."

result = fill_mask(text)

for item in result:
    print(item["sequence"], item["score"])

In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

texts = [
    "The GPU memory is not enough for training BERT.",
    "The football match ended with a dramatic goal.",
    "The recipe requires flour, eggs, and milk."
]

labels = ["machine learning", "sports", "food"]

for text in texts:
    result = classifier(text, candidate_labels=labels)
    print(text)
    print(result["labels"][0], result["scores"][0])
    print("-" * 80)

#An Encoder-Decoder Network for Neural Machine Translation

In [ ]:
from datasets import load_dataset

nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train",
    name="eng-spa",
    split=["validation", "test"]
)

In [ ]:
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)

nmt_train_set = split["train"]
nmt_valid_set = split["test"]

In [ ]:
nmt_train_set[0]

In [ ]:
def train_eng_spa():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

In [ ]:
import tokenizers

max_length = 256
vocab_size = 10_000

nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)

In [ ]:
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)

In [ ]:
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()


In [ ]:
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=[
        "<pad>",
        "<unk>",
        "<s>", # SOS
        "</s>" # EOS
    ]
)

In [ ]:
nmt_tokenizer.train_from_iterator(
    train_eng_spa(),
    trainer=nmt_tokenizer_trainer
)

In [ ]:
print(nmt_tokenizer.encode("I like soccer").ids)

In [ ]:

print(nmt_tokenizer.encode("<s> Me gusta el fútbol").ids)

In [ ]:
print(nmt_tokenizer.token_to_id("<pad>"))

In [ ]:
print(nmt_tokenizer.token_to_id("<unk>"))

In [ ]:
print(nmt_tokenizer.token_to_id("<s>"))

In [ ]:
print(nmt_tokenizer.token_to_id("</s>"))

In [ ]:
from collections import namedtuple

fields = [
    "src_token_ids", #eng
    "src_mask",
    "tgt_token_ids", #spa
    "tgt_mask"
]

class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(
            self.src_token_ids.to(device),
            self.src_mask.to(device),
            self.tgt_token_ids.to(device),
            self.tgt_mask.to(device)
        )

In [ ]:
# src_token_ids → English token IDs
# src_mask      → English attention mask
# tgt_token_ids → Spanish decoder input token IDs
# tgt_mask      → Spanish attention mask

In [ ]:
import torch
from torch.utils.data import DataLoader

In [ ]:
def nmt_collate_fn(batch):
    src_texts = [pair["source_text"] for pair in batch ]

    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch ]

    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)

    src_token_ids = torch.tensor( [enc.ids for enc in src_encodings], dtype=torch.long )

    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings],  dtype=torch.long  )

    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings], dtype=torch.long )

    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings], dtype=torch.long)


    inputs = NmtPair(
        src_token_ids,
        src_mask,
        tgt_token_ids[:, :-1],  # [<s> i love movies </s>] => [<s> i love movies ]
        tgt_mask[:, :-1]
    )

    labels = tgt_token_ids[:, 1:] #[<s> i love movies </s>] => [ i love movies </s>]

    return inputs, labels

In [ ]:
batch_size = 32

nmt_train_loader = DataLoader(
    nmt_train_set,
    batch_size=batch_size,
    collate_fn=nmt_collate_fn,
    shuffle=True
)

nmt_valid_loader = DataLoader(
    nmt_valid_set,
    batch_size=batch_size,
    collate_fn=nmt_collate_fn
)

nmt_test_loader = DataLoader(
    nmt_test_set,
    batch_size=batch_size,
    collate_fn=nmt_collate_fn
)

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

In [ ]:
class NmtModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        pad_id=0,
        hidden_dim=512,
        n_layers=2
    ):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.encoder = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True
        )

        self.decoder = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True
        )

        self.output = nn.Linear(
            in_features=hidden_dim,
            out_features=vocab_size
        )

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)

        src_lengths = pair.src_mask.sum(dim=1)

        src_packed = pack_padded_sequence(
            src_embeddings,
            lengths=src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden_states = self.encoder(src_packed)

        outputs, _ = self.decoder(
            tgt_embeddings,
            hidden_states
        )

        logits = self.output(outputs)

        return logits.permute(0, 2, 1) #(batch_size, tgt_len, vocab_size) but we need (batch_size, vocab_size, tgt_len)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)

vocab_size = nmt_tokenizer.get_vocab_size()

nmt_model = NmtModel(vocab_size).to(device)

print(nmt_model)

In [ ]:
batch, labels = next(iter(nmt_train_loader))

batch = batch.to(device)
labels = labels.to(device)

logits = nmt_model(batch)

print("logits shape:", logits.shape)
print("labels shape:", labels.shape)

In [ ]:
xentropy = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
optimizer = torch.optim.NAdam(
    nmt_model.parameters(),
    lr=3e-4
)

In [ ]:
def token_accuracy(logits, labels, pad_id=0):
    predictions = logits.argmax(dim=1)

    mask = labels != pad_id

    correct = (predictions == labels) & mask

    return correct.sum().float() / mask.sum().float()

In [ ]:
from tqdm.auto import tqdm



def train_one_epoch(model, dataloader, optimizer, loss_fn, device, pad_id=0):
    model.train()

    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    progress_bar = tqdm(dataloader, desc="Training")

    for batch, labels in progress_bar:
        batch = batch.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(batch)

        loss = loss_fn(logits, labels)
        acc = token_accuracy(logits, labels, pad_id=pad_id)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += acc.item()
        total_batches += 1

        progress_bar.set_postfix({
            "loss": loss.item(),
            "acc": acc.item()
        })

    return total_loss / total_batches, total_acc / total_batches


def evaluate(model, dataloader, loss_fn, device, pad_id=0):
    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    progress_bar = tqdm(dataloader, desc="Validation")

    with torch.no_grad():
        for batch, labels in progress_bar:
            batch = batch.to(device)
            labels = labels.to(device)

            logits = model(batch)

            loss = loss_fn(logits, labels)
            acc = token_accuracy(logits, labels, pad_id=pad_id)

            total_loss += loss.item()
            total_acc += acc.item()
            total_batches += 1

            progress_bar.set_postfix({
                "val_loss": loss.item(),
                "val_acc": acc.item()
            })

    return total_loss / total_batches, total_acc / total_batches


num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        nmt_model,
        nmt_train_loader,
        optimizer,
        xentropy,
        device
    )

    valid_loss, valid_acc = evaluate(
        nmt_model,
        nmt_valid_loader,
        xentropy,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.4f}"
    )


In [ ]:
def translate(
    model,
    src_text,
    max_length=20,
    pad_id=0,
    eos_id=3
):
    model.eval()

    tgt_text = ""

    for index in range(max_length):

        batch, _ = nmt_collate_fn([
            {
                "source_text": src_text,
                "target_text": tgt_text
            }
        ])

        batch = batch.to(device)

        with torch.no_grad():
            y_logits = model(batch)

        y_token_ids = y_logits.argmax(dim=1)

        next_token_id = y_token_ids[0, index].item()

        next_token = nmt_tokenizer.id_to_token(next_token_id)

        tgt_text += " " + next_token

        if next_token_id == eos_id:
            break

    return tgt_text




In [ ]:
nmt_model.eval()


In [ ]:
translate(
    nmt_model,
    "I like to play soccer with my friends."
)

In [ ]:
translate(
    nmt_model,
    "Hi , my friend"
)


#Beam Search

In [ ]:
import torch.nn.functional as F

In [ ]:
def make_nmt_pair_for_generation(src_text, tgt_token_ids, device):
    src_encoding = nmt_tokenizer.encode(src_text)

    src_token_ids = torch.tensor(
        [src_encoding.ids],
        dtype=torch.long,
        device=device
    )

    src_mask = torch.tensor(
        [src_encoding.attention_mask],
        dtype=torch.long,
        device=device
    )

    tgt_token_ids = torch.tensor(
        [tgt_token_ids],
        dtype=torch.long,
        device=device
    )

    tgt_mask = torch.ones_like(tgt_token_ids)

    return NmtPair(
        src_token_ids=src_token_ids,
        src_mask=src_mask,
        tgt_token_ids=tgt_token_ids,
        tgt_mask=tgt_mask
    )

In [ ]:
def beam_search_translate(
    model,
    src_text,
    beam_width=3,
    max_length=20,
    device=device
):
    model.eval()

    sos_id = nmt_tokenizer.token_to_id("<s>")
    eos_id = nmt_tokenizer.token_to_id("</s>")


    beams = [
        ([sos_id], 0.0, False)
    ]

    with torch.no_grad():
        for step in range(max_length):
            all_candidates = []

            for token_ids, score, finished in beams:


                if finished:
                    all_candidates.append(
                        (token_ids, score, finished)
                    )
                    continue

                pair = make_nmt_pair_for_generation(
                    src_text=src_text,
                    tgt_token_ids=token_ids,
                    device=device
                )

                logits = model(pair)

                next_token_logits = logits[0, :, -1]

                log_probs = F.log_softmax(
                    next_token_logits,
                    dim=-1
                )


                top_log_probs, top_token_ids = torch.topk(
                    log_probs,
                    k=beam_width
                )

                for log_prob, next_token_id in zip(top_log_probs, top_token_ids):
                    next_token_id = next_token_id.item()

                    new_token_ids = token_ids + [next_token_id]
                    new_score = score + log_prob.item()
                    new_finished = next_token_id == eos_id

                    all_candidates.append(
                        (new_token_ids, new_score, new_finished)
                    )

            all_candidates = sorted(
                all_candidates,
                key=lambda x: x[1],
                reverse=True
            )

            beams = all_candidates[:beam_width]

            if all(finished for _, _, finished in beams):
                break

    best_token_ids, best_score, finished = beams[0]

    best_token_ids = best_token_ids[1:]

    if eos_id in best_token_ids:
        eos_index = best_token_ids.index(eos_id)
        best_token_ids = best_token_ids[:eos_index]

    translation = nmt_tokenizer.decode(
        best_token_ids,
        skip_special_tokens=True
    )

    return translation

In [ ]:
beam_search_translate(
    nmt_model,
    "I like to play soccer with my friends.",
    beam_width=3,
    max_length=20
)

#Attention Mechanisms

In [ ]:
from torch.nn.utils.rnn import pad_packed_sequence, pack_padded_sequence

In [ ]:
def attention(query, key, value):
    scores = query @ key.transpose(1, 2)
    weights = torch.softmax(scores, dim=-1)
    return weights @ value

In [ ]:
class NmtAttentionModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        pad_id=0,
        hidden_dim=512,
        n_layers=2
    ):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.encoder = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True
        )

        self.decoder = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True
        )

        self.output = nn.Linear(
            in_features=2 * hidden_dim,
            out_features=vocab_size
        )

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)

        src_lengths = pair.src_mask.sum(dim=1)

        src_packed = pack_padded_sequence(
            src_embeddings,
            lengths=src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        encoder_outputs_packed, hidden_states = self.encoder(src_packed)

        encoder_outputs, _ = pad_packed_sequence(
            encoder_outputs_packed,
            batch_first=True
        )

        decoder_outputs, _ = self.decoder(
            tgt_embeddings,
            hidden_states
        )

        attn_output = attention(
            query=decoder_outputs,
            key=encoder_outputs,
            value=encoder_outputs
        )

        combined_output = torch.cat(
            (attn_output, decoder_outputs),
            dim=-1
        )

        logits = self.output(combined_output)

        return logits.permute(0, 2, 1)

In [ ]:
nmt_model = NmtModel(vocab_size).to(device)

In [ ]:
nmt_attn_model = NmtAttentionModel(vocab_size).to(device)

In [ ]:
optimizer = torch.optim.NAdam(
    nmt_attn_model.parameters(),
    lr=3e-4
)

In [ ]:
xentropy = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        nmt_attn_model,
        nmt_train_loader,
        optimizer,
        xentropy,
        device
    )

    valid_loss, valid_acc = evaluate(
        nmt_attn_model,
        nmt_valid_loader,
        xentropy,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.4f}"
    )

In [ ]:
translate(
    nmt_attn_model,
    "I like to play soccer with my friend after school because it helps us stay active and have fun together"
)

In [ ]:
beam_search_translate(
    nmt_attn_model,
    "I like to play soccer with my friend after school because it helps us stay active and have fun together",
    beam_width=3,
    max_length=20
)